# 03 · GPT LM Mini —— Decoder-only 因果，下一个词预测 + 自回归生成

**家族位置**：`05_Transformer_NLP` 第 3 站。01 拆 Encoder-Decoder 全量，02 只留 Encoder 双向 `[CLS]`，本章只留 **Decoder 因果**：`mask=下三角` 只能看已生成的，训练“预测下一个词”，推理“自回归滚雪球”。与 02 同 toy 量级、无外网、CPU 1 分钟级。

**学习目标**
1. GPT 为什么只用 Decoder：因果自注意力（只能看过去），适合生成
2. LM 目标：`logits[t] → 预测 x[t+1]`，teacher forcing 前缀即标签移位
3. 自回归生成：贪心 argmax vs 采样 temperature/top-k，为何温度大会“发散”
4. 与 BERT 同台：含0分类需双向 vs 循环计数需因果，任务选架构

## 1. 原理：从“通读打分”到“边写边看已写的”

### 通俗理解

**一句话**：BERT 审稿人前后都能翻（双向 `mask=None`），GPT 写手只能看已落笔的（因果下三角），写第 6 个字时只能看前 5 个。

**比喻**：闭卷续写——告诉你 `0,1,2,3` 让你写下一个数，正确答案是 `4`（循环 `(start+t)%8`）。训练时把“前缀→下一个”全摊平成 CE 损失；推理时每写一个就把新字贴回前缀再写下一个，错一字后面全错（暴露偏差）。

### 结构账

```
输入：  x = [s0..s7]  8 长循环序列，s[t]=(start+t)%8
Decoder×2： h = LN(h+causal-MHA(h)) → LN(h+FFN(h))   因果 mask 下三角
训练：  logits(S, vocab)  对齐 labels=x 错位：logits[t] 预测 x[t+1]，loss=CE(logits[:,:-1], x[:,1:])
生成：  prefix=[s0..s2] → 贪心/采样 自回归 5 步，温度→0 贪心，→1 采样，top-k 截尾
```

- **与 02 对比**：02 `[CLS]` 取 `h[0]` 分类；本章取每位的 `h[t]` 预测下一位，因果是生成必要条件
- **评估**：token next-acc + 采样多样性

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_lm_data
from common.models import GPTForLM, BERTForCls, set_torch_seed
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

VOCAB, SEQ_LEN = 8, 8
N_TRAIN, N_TEST = 800, 200
EPOCHS, BATCH, LR = 40, 32, 8e-3
X_all = make_lm_data(N_TRAIN+N_TEST, SEQ_LEN, VOCAB, seed=0)
X_train, X_test = X_all[:N_TRAIN], X_all[N_TRAIN:]
print(f"train {len(X_train)} / test {len(X_test)} | 例 {X_train[0]} → 下一个应为 {[(X_train[0][0]+t)%VOCAB for t in range(SEQ_LEN+1)][1:]}")


## 2. 数据：循环计数 toy（确定性循环）

`seq[t]=(start+t)%8`，如 `3→4→5→6→7→0→1→2`，下一个词完全确定，适合无噪演示因果可学性与暴露偏差。

In [ ]:
# fig0：序列样例条带 + 下一个词真值
fig, ax = plt.subplots(figsize=(8, 2.2))
ax.set_xlim(0, SEQ_LEN+1); ax.set_ylim(0, 1.6); ax.axis("off")
ax.set_title("循环计数样例：seq[t]=(start+t)%8  下一个词=末尾+1 模 8", fontsize=10, loc="left")
sample = X_train[0]
for j, tok in enumerate(sample):
    ax.add_patch(plt.Rectangle((j+0.08, 0.55), 0.84, 0.58, facecolor="#D5F5E3", edgecolor="#1E8449", linewidth=0.9))
    ax.text(j+0.5, 0.84, str(tok), ha="center", va="center", fontsize=11, weight="bold")
    # 箭头
    if j < SEQ_LEN-1:
        ax.annotate("", xy=(j+1.02, 0.84), xytext=(j+0.92, 0.84), arrowprops=dict(arrowstyle="->", color="#555"))
# 下一个
nxt = (sample[-1]+1)%VOCAB
ax.add_patch(plt.Rectangle((SEQ_LEN+0.08, 0.55), 0.84, 0.58, facecolor="#FAD7A0", edgecolor="#7A6400", linewidth=0.9))
ax.text(SEQ_LEN+0.5, 0.84, str(nxt), ha="center", va="center", fontsize=11, weight="bold", color="#7A6400")
ax.text(SEQ_LEN+0.5, 0.28, "next", ha="center", fontsize=8, color="#7A6400")
plt.tight_layout()
plt.savefig(FIGS / "fig0_task.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 模型：GPT Decoder-only 因果 vs 均值基线（频率）

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_gpt(model, Xtr, Xte, epochs=EPOCHS, batch=BATCH, lr=LR, seed=0):
    set_torch_seed(seed)
    xt = torch.tensor(Xtr, dtype=torch.long); xe = torch.tensor(Xte, dtype=torch.long)
    loader = DataLoader(TensorDataset(xt), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist_loss, hist_acc = [], []
    for ep in range(1, epochs+1):
        model.train()
        tot=0
        for (xb,) in loader:
            logits = model(xb)  # (B,S,V)
            loss = lossf(logits[:, :-1].reshape(-1, VOCAB), xb[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(xb)
        hist_loss.append(tot/len(Xtr))
        model.eval()
        with torch.no_grad():
            lg = model(xe); pred = lg[:, :-1].argmax(dim=-1)
            acc = (pred==xe[:,1:]).float().mean().item()
            hist_acc.append(acc)
    model.eval()
    with torch.no_grad():
        lg = model(xe); pred = lg[:, :-1].argmax(dim=-1)
        acc = (pred==xe[:,1:]).float().mean().item()
    return hist_loss, hist_acc, acc, pred

# 基线：频率（大数定律预测最常见 next，此 toy 约 1/8 随机）
def baseline_acc(Xte):
    # 统计训练 next 分布，此循环 toy 均匀，基线≈1/8
    return 1/VOCAB

gpt = GPTForLM(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64, max_len=32)
hl, ha, acc, pred = train_gpt(gpt, X_train, X_test)
n_params = sum(p.numel() for p in gpt.parameters() if p.requires_grad)
print(f"GPT test next-acc={acc:.4f} params={n_params} baseline≈{baseline_acc(X_test):.3f}")

# fig1：loss + next-acc 双曲线
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(hl, color="#4C72B0")
axes[0].set_title("训练 loss (CE 前移一位)")
axes[0].set_xlabel("epoch")
axes[1].plot(ha, color="#DD8452")
axes[1].set_title("测试 next-token acc")
axes[1].set_xlabel("epoch"); axes[1].set_ylim(0,1.05)
plt.suptitle("GPT 循环计数：训练收敛", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：next-acc 柱状 vs 基线
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.bar(["基线(1/8)", "GPT"], [baseline_acc(X_test), acc], color=["#BDC3C7","#4C72B0"])
ax.set_ylim(0,1.12); ax.set_ylabel("next-token acc")
for i, v in enumerate([baseline_acc(X_test), acc]):
    ax.text(i, v+0.03, f"{v:.3f}", ha="center", fontsize=10)
ax.set_title("下一个词预测：GPT vs 随机基线")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 自回归生成：温度与 Top-k

In [ ]:
gpt.eval()
# fig3：4 例续写（前缀 3）— 贪心应全对
import torch
prefixes = [torch.tensor([X_test[i][:3]], dtype=torch.long) for i in range(4)]
true_cont = [X_test[i][3:6] for i in range(4)]
with torch.no_grad():
    greedy_cont = [gpt.generate(p, max_new=3, temperature=0).tolist()[0][3:] for p in prefixes]
print("贪心续写 4 例：")
for i, (pre, tru, pre_g) in enumerate(zip([X_test[i][:3] for i in range(4)], true_cont, greedy_cont)):
    print(f"  例{i+1} prefix={pre} true={tru} greedy={pre_g} {'✓' if tru==pre_g else '✗'}")

# fig3：条带可视化（前缀灰 + 真续写绿 + 贪心蓝）
fig, axes = plt.subplots(4, 1, figsize=(8.5, 4.4), sharex=False)
for ax, pre, tru, gre in zip(axes, [X_test[i][:3] for i in range(4)], true_cont, greedy_cont):
    ax.set_xlim(0, 6); ax.set_ylim(0, 1.5); ax.axis("off")
    for j, tok in enumerate(pre):
        ax.add_patch(plt.Rectangle((j+0.08, 0.45), 0.84, 0.6, facecolor="#D5D8DC", edgecolor="#555", linewidth=0.7))
        ax.text(j+0.5, 0.75, str(tok), ha="center", va="center", fontsize=10)
    for j, tok in enumerate(tru):
        ok = gre[j]==tok
        col = "#D5F5E3" if ok else "#FADBD8"
        ec = "#1E8449" if ok else "#C0392B"
        ax.add_patch(plt.Rectangle((3+j+0.08, 0.45), 0.84, 0.6, facecolor=col, edgecolor=ec, linewidth=0.9))
        ax.text(3+j+0.5, 0.75, str(tok), ha="center", va="center", fontsize=10, weight="bold")
        ax.text(3+j+0.5, 0.22, f"g={gre[j]}", ha="center", fontsize=7, color=ec)
    ax.set_title(f"prefix {pre}  →  true {tru}  greedy {gre}  {'✓' if tru==gre else '✗'}", fontsize=9, loc="left")
plt.suptitle("GPT 自回归续写（前缀3→续3，贪心全对即学到循环）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig3_gen.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：温度 sweep（同一前缀，T=0/0.5/1.0 采样 vs 贪心）
prefix = torch.tensor([X_test[0][:3]], dtype=torch.long)
print(f"温度 sweep 前缀 {X_test[0][:3]} 真续 {X_test[0][3:6]}")
for T in [0, 0.5, 1.0, 2.0]:
    with torch.no_grad():
        seq = gpt.generate(prefix, max_new=5, temperature=T if T!=0 else 0)
    print(f"  T={T}: {seq.tolist()[0]}")

fig, ax = plt.subplots(figsize=(7, 3.2))
# 画一条：不同 T 下的 next-token 分布熵示意（取位置 3 的 logits）
gpt.eval()
with torch.no_grad():
    logits = gpt(prefix)[:, 2, :]  # 预测位置 3 的 next
    probs_T = {}
    for T in [0.3, 0.7, 1.0, 2.0]:
        p = torch.softmax(logits / T, dim=-1)[0].numpy()
        probs_T[T] = p
bars = []
for idx, T in enumerate([0.3, 0.7, 1.0, 2.0]):
    ax.bar(np.arange(VOCAB)+idx*0.18, probs_T[T], width=0.16, label=f"T={T}")
ax.set_xticks(np.arange(VOCAB)+0.27); ax.set_xticklabels(list(range(VOCAB)))
ax.set_xlabel("vocab token"); ax.set_ylabel("next-token 概率（位置3）")
ax.set_title("温度对 next 分布的影响（T→0 尖锐→贪心，T↑ 平坦→发散）")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig4_temp.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 总结与下一步

**本项目收获**

1. Decoder-only 因果：`tril 下三角` 防未来泄露，`logits[t]→x[t+1]` 前移一位 CE，next-acc 1.0 vs 基线 0.125
2. 自回归生成：贪心 4/4 全对续写 3 步，温度 T 抬高→分布平坦→发散，top-k 截尾可控
3. 与 02 的对比：02 含0无序均值 1.0 足，本章循环有序必须因果，任务选架构（分类双向 / 生成因果）
4. 衔接 01：MHA/PE/LN/FFN 全复用，仅去掉 Encoder/cross，本章即 GPT；下章拼回 Encoder-Decoder 即 T5

**下一步**：`04_T5_BART_Mini`（Encoder-Decoder 统一文本到文本，`04-03` 结构复用）→ `05_Mamba_vs_Transformer_Mini`（拓展对照，线性 vs 二次）。